In [ ]:
import pandas as pd


def reshape_results_to_wide_format(df: pd.DataFrame) -> pd.DataFrame:
    """
    Transforms a long-format DataFrame of results into a wide-format DataFrame.

    The resulting DataFrame will have 'method' as the index, and a single column
    for each combination of 'dataset' and 'metric'.

    Args:
        df (pd.DataFrame): A DataFrame with columns ['method', 'dataset', 'metric', 'value'].

    Returns:
        pd.DataFrame: The reshaped, wide-format DataFrame.
    """
    pivoted_df = df.pivot_table(
        index='method',
        columns=['dataset', 'metric'],
        values='value'
    )

    pivoted_df.columns = [f"{col[0]}_{col[1]}" for col in pivoted_df.columns]
    return pivoted_df

In [ ]:
import pickle
import numpy as np

rf_datasets = {
    'Linear': 'synthetic_data/herg_ecfp_linear',
    'Piecewise': 'synthetic_data/herg_ecfp_piecewise',
    'Polynomial': 'synthetic_data/herg_ecfp_nonlinear',
    'CNOHF': 'cnohf_data/cnohf_ecfp',
    'COF': 'cof_data/cof_ecfp_descriptor',
    'Photoswitch': 'photoswitch_data/photoswitch_ecfp',
    'Polymers': 'polymers_data/polymers_ecfp',
    'Redox': 'redox_data/redox_ecfp',
}

pred_models_results = {
    'Dataset': [],
    'RMSE': [],
    'RMSE std': [],
    'SMAPE': [],
    'SMAPE std': [],
    'PA': [],
    'PA std': [],
}

for dname, dataset in rf_datasets.items():
    with open(f'../results/{dataset}/results.pickle', 'rb') as f:
        results = pickle.load(f)
    pred_models_results['Dataset'].append(dname)

    m_mean = {}
    m_std = {}
    for method in results['scores']:
        m_mean[method] = [np.mean(results['scores'][method])]
        m_std[method] = [np.std(results['scores'][method])]
    pred_models_results['RMSE'].append(m_mean['rmse'][0])
    pred_models_results['RMSE std'].append(m_std['rmse'][0])
    pred_models_results['SMAPE'].append(m_mean['smape'][0])
    pred_models_results['SMAPE std'].append(m_std['smape'][0])
    pred_models_results['PA'].append(m_mean['pairwise_accuracy_score'][0])
    pred_models_results['PA std'].append(m_std['pairwise_accuracy_score'][0])

df_pred = pd.DataFrame(pred_models_results)

mean_cols = ['RMSE', 'SMAPE', 'PA']
std_cols = ['RMSE std', 'SMAPE std', 'PA std']
df_mean_str = df_pred[mean_cols].map('{:.2f}'.format)
df_std_str = df_pred[std_cols].map('{:.2f}'.format)
df_std_str.rename(columns={
    'RMSE std': 'RMSE',
    'SMAPE std': 'SMAPE',
    'PA std': 'PA',
}, inplace=True)

df_pred_str = df_mean_str + '(' + df_std_str + ')'
df_pred_str.insert(0, 'Dataset', df_pred['Dataset'])
df_pred_str

df_pred_str.set_index('Dataset', inplace=True)

print(df_pred_str.style.format(precision=2).to_latex())


In [ ]:
gt_datasets = {
    'GT Linear': 'gt_synthetic_data/herg_ecfp_linear',
    'GT Piecewise': 'gt_synthetic_data/herg_ecfp_piecewise',
    'GT Polynomial': 'gt_synthetic_data/herg_ecfp_nonlinear',
}

metrics = {}
cf_validity = {}
cf_similarity = {}
reference_lists = {}

for dname, dataset in gt_datasets.items():
    with open(f'../results/{dataset}/explanations/analysis3/metrics_results.pickle', 'rb') as f:
        results = pickle.load(f)
    metrics[dname] = results
    with open(f'../results/{dataset}/explanations/analysis3/cf_validity_results.pickle', 'rb') as f:
        results = pickle.load(f)
    cf_validity[dname] = results
    with open(f'../results/{dataset}/explanations/analysis3/cf_similarity_results.pickle', 'rb') as f:
        results = pickle.load(f)
    cf_similarity[dname] = results
    with open(f'../results/{dataset}/explanations/analysis3/reference_lists.pickle', 'rb') as f:
        results = pickle.load(f)
    results = [rr for r in results for rr in r]
    reference_lists[dname] = results

for dname, dataset in rf_datasets.items():
    try:
        with open(f'../results/{dataset}/explanations/analysis3/metrics_results.pickle', 'rb') as f:
            results = pickle.load(f)
        metrics[dname] = results
        with open(f'../results/{dataset}/explanations/analysis3/cf_validity_results.pickle', 'rb') as f:
            results = pickle.load(f)
        cf_validity[dname] = results
        with open(f'../results/{dataset}/explanations/analysis3/cf_similarity_results.pickle', 'rb') as f:
            results = pickle.load(f)
        cf_similarity[dname] = results
        with open(f'../results/{dataset}/explanations/analysis3/reference_lists.pickle', 'rb') as f:
            results = pickle.load(f)
        results = [rr for r in results for rr in r]
        reference_lists[dname] = results
    except:
        print(f"Results not found for dataset: {dname}")
metrics

In [ ]:
df_validity = {'method': [], 'dataset': [], 'metric': [], 'value': []}
for k in cf_validity:
    for method in cf_validity[k]:
        df_validity['method'].append(method)
        df_validity['dataset'].append(k)
        df_validity['metric'].append('validity')
        df_validity['value'].append(cf_validity[k][method])
df_validity = pd.DataFrame(df_validity)
df_validity = reshape_results_to_wide_format(df_validity)

df_validity = df_validity.T
df_validity.index = [c.replace('_validity', '') for c in df_validity.index]
df_validity = df_validity[['mmace', 'meg']]
print(df_validity.style.format(precision=2).to_latex())

In [ ]:
df_similarity = {'method': [], 'dataset': [], 'metric': [], 'value': []}
for k in cf_similarity:
    for method in cf_similarity[k]:
        for i, m in enumerate(['similarity', 'similarity std']):
            df_similarity['dataset'].append(k)
            df_similarity['method'].append(method)
            df_similarity['metric'].append(m)
            df_similarity['value'].append(cf_similarity[k][method][i])
df_similarity = pd.DataFrame(df_similarity)
df_similarity = reshape_results_to_wide_format(df_similarity)
df_similarity_mean = df_similarity[[c for c in df_similarity.columns if 'similarity' in c and 'std' not in c]]
df_similarity_std = df_similarity[[c for c in df_similarity.columns if 'similarity std' in c]]
df_similarity_std.rename(columns={d: d.replace('similarity std', 'similarity') for d in df_similarity_std.columns}, inplace=True)

df_mean_str = df_similarity_mean.map('{:.2f}'.format)
df_std_str = df_similarity_std.map('{:.2f}'.format)
df_similarity = df_mean_str + '(' + df_std_str + ')'
df_similarity = df_similarity.T
df_similarity.index = [c.replace('_similarity', '') for c in df_similarity.index]
df_similarity = df_similarity[['mmace', 'meg']]
print(df_similarity.style.format(precision=2).to_latex())

In [ ]:
def _transform_dataframe(df):
    metrics_dataframe = pd.DataFrame(df)
    metrics_dataframe = reshape_results_to_wide_format(metrics_dataframe)

    metrics_df_mean = metrics_dataframe[[
        c for c in metrics_dataframe.columns if 'std' not in c
    ]]
    metrics_df_std = metrics_dataframe[[
        c for c in metrics_dataframe.columns if 'std' in c
    ]]

    df_mean_str = metrics_df_mean.map('{:.2f}'.format)
    df_std_str = metrics_df_std.map('{:.2f}'.format)
    df_mean_str.rename(columns={d: d.replace('_mean', '') for d in df_mean_str.columns}, inplace=True)
    df_std_str.rename(columns={d: d.replace('_std', '') for d in df_std_str.columns}, inplace=True)

    # Concatenate the string DataFrames
    # This works element-wise because the index and columns match perfectly
    df_combined = df_mean_str + '(' + df_std_str + ')'
    df_combined.rename(columns={d: d.replace('_', ' ') for d in df_combined.columns
    }, inplace=True)
    metrics_df_mean.rename(columns={d: d.replace('_mean', '') for d in metrics_df_mean.columns}, inplace=True)
    return df_combined, metrics_df_mean.rename(columns={d: d.replace('_', ' ') for d in metrics_df_mean.columns})


def get_metrics(datasets, metrics_dict, metrics, reference_lists):
    metrics_dataframe = {'method': [], 'dataset': [], 'metric': [], 'value': []}
    metrics_intersection_dataframe = {'method': [], 'dataset': [], 'metric': [], 'value': []}
    change_name_dict = {
        'pgis_raw': 'pgi',
        'pgus_raw': 'pgu',
        'fa': 'fa'
    }
    for dataset in datasets:
        reference_list = reference_lists[dataset]
        for method, method_results in metrics[dataset].items():
            for m in metrics_dict:
                whole = [p for p in method_results[1][m] if p is not None]
                intersection = [p for i, p in enumerate(method_results[1][m]) if reference_list[i] is not None]
                metrics_dataframe['dataset'].append(dataset)
                metrics_dataframe['method'].append(method)
                metrics_dataframe['metric'].append(f"{change_name_dict[m]}_std")
                metrics_dataframe['value'].append(np.std(whole))
                metrics_dataframe['dataset'].append(dataset)
                metrics_dataframe['method'].append(method)
                metrics_dataframe['metric'].append(f"{change_name_dict[m]}_mean")
                metrics_dataframe['value'].append(np.mean(whole))

                metrics_intersection_dataframe['dataset'].append(dataset)
                metrics_intersection_dataframe['method'].append(method)
                metrics_intersection_dataframe['metric'].append(f"{change_name_dict[m]}_std")
                metrics_intersection_dataframe['value'].append(np.std(intersection))
                metrics_intersection_dataframe['dataset'].append(dataset)
                metrics_intersection_dataframe['method'].append(method)
                metrics_intersection_dataframe['metric'].append(f"{change_name_dict[m]}_mean")
                metrics_intersection_dataframe['value'].append(np.mean(intersection))

    metrics_df, metric_df_mean = _transform_dataframe(metrics_dataframe)
    intersection_df, intersection_df_mean = _transform_dataframe(metrics_intersection_dataframe)
    return metrics_df, metric_df_mean, intersection_df, intersection_df_mean

In [ ]:
# gt results

gt_results = ['GT Linear', 'GT Piecewise', 'GT Polynomial']
metrics_names = ['pgis_raw', 'pgus_raw', 'fa']

df_combined_all, gt_means_all, df_combined_inter, df_mean_inter = get_metrics(gt_results, metrics_names, metrics, reference_lists)
print(df_combined_inter.style.format(precision=3).to_latex())

In [ ]:
gt_means_all

In [ ]:
df_mean_inter

In [ ]:
def plot_pgu_pgi(metrics_mean, datasets, ax):
    methods_params = {
        'LIME': {'marker': 'o', 'color': '#3b4b63', 'size': 100, 'zorder': 5, 'font_weight': 'normal'},
        'SHAP': {'marker': '^', 'color': '#6c4cb6', 'size': 200, 'zorder': 10, 'font_weight': 'normal'},
        'SHAP-IQ-1': {'marker': 'o', 'color': '#3b4b63', 'size': 100, 'zorder': 5, 'font_weight': 'normal'},
        'SHAP-IQ-2': {'marker': 'o', 'color': '#3b4b63', 'size': 100, 'zorder': 5, 'font_weight': 'normal'},
        'MEG': {'marker': 'o', 'color': '#3b4b63', 'size': 100, 'zorder': 5, 'font_weight': 'normal'},
        'MMACE': {'marker': 'o', 'color': '#3b4b63', 'size': 100, 'zorder': 5, 'font_weight': 'normal'},
        'Mean': {'marker': 'D', 'color': '#cfd051', 'size': 200, 'zorder': 10, 'font_weight': 'normal'},
        'RRA': {'marker': '*', 'color': '#cfd051', 'size': 200, 'zorder': 10, 'font_weight': 'normal'},
    }
    rename_methods = {
        'lime': 'LIME',
        'shap': 'SHAP',
        'shapiq1': 'SHAP-IQ-1',
        'shapiq2': 'SHAP-IQ-2',
        'meg': 'MEG',
        'mmace': 'MMACE',
        'aggregated_mean': 'Mean',
        'aggregated_rra': 'RRA'
    }
    jitter_strength = 0.001

    for i, dataset in enumerate(datasets):
        pgu = metrics_mean[f"{dataset} pgu"]
        pgi = metrics_mean[f"{dataset} pgi"]

        pgu.index = pgu.index.map(rename_methods)
        pgi.index = pgi.index.map(rename_methods)

        x_span = pgu.max() - pgu.min()
        y_span = pgi.max() - pgi.min()
        x_span = x_span if x_span > 0 else 1.0
        y_span = y_span if y_span > 0 else 1.0

        for method in pgu.index:
            x = pgu[method] + np.random.uniform(-1, 1) * x_span * jitter_strength
            y = pgi[method] + np.random.uniform(-1, 1) * y_span * jitter_strength
            params = methods_params[method]
            if method in ['SHAP', 'Mean', 'RRA']:
                alpha = 1.0
            else:
                alpha = 0.6
            ax[i].scatter(x, y, marker=params['marker'], c=params['color'], s=params['size'],
                       edgecolors='#3b4b63', linewidth=0.5, zorder=params['zorder'], alpha=alpha)
            ax[i].text(x, y, f"{method}", fontsize=11, fontweight=params['font_weight'], zorder=params['zorder']+1)
    return ax


In [ ]:
real_rf_results = ['CNOHF', 'Photoswitch', 'Polymers', 'Redox', 'COF']
metrics_names = ['pgis_raw', 'pgus_raw']
df_combined_all, gt_means_all, df_combined_inter, df_mean_inter = get_metrics(real_rf_results, metrics_names, metrics, reference_lists)
df_combined_inter = df_combined_inter[[
    'CNOHF pgi', 'CNOHF pgu',
    'Photoswitch pgi', 'Photoswitch pgu',
    'Polymers pgi', 'Polymers pgu',
    'Redox pgi', 'Redox pgu',
    'COF pgi', 'COF pgu',
]]
print(df_combined_all.style.format(precision=3).to_latex())

In [ ]:
syn_rf_results = ['Linear', 'Piecewise', 'Polynomial']
metrics_names = ['pgis_raw', 'pgus_raw']
df_combined_all, gt_means_all, df_combined_inter, df_mean_inter = get_metrics(syn_rf_results, metrics_names, metrics, reference_lists)
df_combined_inter = df_combined_inter[[
    'Linear pgi', 'Linear pgu',
    'Piecewise pgi', 'Piecewise pgu',
    'Polynomial pgi', 'Polynomial pgu',
]]
print(df_combined_all.style.format(precision=3).to_latex())

In [ ]:
from src.analysis.xai_eval import rank_correlation
from scripts.bulk_metric_compute_gt import convert_term_ranking_to_feature_ranking

gt_datasets = {
    'GT Linear': 'gt_synthetic_data/herg_ecfp_linear',
    'GT Piecewise': 'gt_synthetic_data/herg_ecfp_piecewise',
    'GT Polynomial': 'gt_synthetic_data/herg_ecfp_nonlinear',
}

gt_correlations = {}

for dname, dataset in gt_datasets.items():
    with open(f'../results/{dataset}/explanations/analysis/ranking_per_fold_results.pickle', 'rb') as f:
        results = pickle.load(f)
    pairs_of_ranks = [(key1, key2) for key1 in results.keys() for key2 in results.keys()]
    correla = {}
    for key1, key2 in pairs_of_ranks:
        corrs = []
        for ii in range(len(results[key1])):
            r1 = results[key1][ii]
            r2 = results[key2][ii]
            if 'aggregated' in key1:
                r1 = pd.DataFrame({
                    'features': r1,
                    'abs_ranking': [-v for v in range(1, len(r1) + 1)],
                    'rank': np.arange(1, len(r1) + 1)
                })
            if 'aggregated' in key2:
                r2 = pd.DataFrame({
                    'features': r2,
                    'abs_ranking': [-v for v in range(1, len(r2) + 1)],
                    'rank': np.arange(1, len(r2) + 1)
                })
            rank1 = convert_term_ranking_to_feature_ranking(r1)
            rank2 = convert_term_ranking_to_feature_ranking(r2)
            c = rank_correlation(rank1, rank2, k=4).statistic
            corrs.append(c)
        correlation = np.mean(corrs)
        correla[(key1, key2)] = correlation
    print(dname, dataset)
    gt_correlations[dname] = correla

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(1, 3, figsize=(15, 5), sharey=True, sharex=True)
name_data_mapping = {
    'GT Linear': 'Linear',
    'GT Piecewise': 'Piecewise',
    'GT Polynomial': 'Polynomial'
}
order_mapping = {
    'GT Linear': 0,
    'GT Piecewise': 1,
    'GT Polynomial': 2
}
methods_mapping = {
    'lime': 'LIME',
    'shap': 'SHAP',
    'shapiq1': 'SHAP-IQ-1',
    'shapiq2': 'SHAP-IQ-2',
    'meg': 'MEG',
    'mmace': 'MMACE',
    'aggregated_mean': 'Mean',
    'aggregated_rra': 'RRA'
}
methods_order = ['LIME', 'SHAP', 'SHAP-IQ-1', 'SHAP-IQ-2', 'MMACE', 'MEG', 'Mean', 'RRA']
plt.style.use('default')
for i, (key, data) in enumerate(gt_correlations.items()):
    if key not in name_data_mapping:
        continue
    s = pd.Series(data)
    correlation_matrix = s.unstack()
    renamed_matrix = correlation_matrix.rename(index=methods_mapping, columns=methods_mapping)
    display(renamed_matrix)
    renamed_matrix = renamed_matrix[methods_order].loc[methods_order]
    mappable = sns.heatmap(renamed_matrix, annot=True, fmt=".1f", ax=ax[order_mapping[key]], cmap='coolwarm', vmin=-1, vmax=1, cbar=False, annot_kws={'size': 13})

    ax[order_mapping[key]].set_title(name_data_mapping[key], fontsize=16)
    ax[order_mapping[key]].tick_params(axis='both', which='major', labelsize=14)

plt.savefig('../results/correlation_matrices_gt.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
from src.analysis.xai_eval import rank_correlation
from scripts.bulk_metric_compute_gt import convert_term_ranking_to_feature_ranking

rf_datasets = {
    'Linear': 'synthetic_data/herg_ecfp_linear',
    'Piecewise': 'synthetic_data/herg_ecfp_piecewise',
    'Polynomial': 'synthetic_data/herg_ecfp_nonlinear',
    'CNOHF': 'cnohf_data/cnohf_ecfp',
    'COF': 'cof_data/cof_ecfp_descriptor',
    'Photoswitch': 'photoswitch_data/photoswitch_ecfp',
    'Polymers': 'polymers_data/polymers_ecfp',
    'Redox': 'redox_data/redox_ecfp',
}
rf_correlations = {}

for dname, dataset in rf_datasets.items():
    with open(f'../results/{dataset}/explanations/analysis/ranking_per_fold_results.pickle', 'rb') as f:
        results = pickle.load(f)
    pairs_of_ranks = [(key1, key2) for key1 in results.keys() for key2 in results.keys()]
    correla = {}
    for key1, key2 in pairs_of_ranks:
        corrs = []
        for ii in range(len(results[key1])):
            r1 = results[key1][ii]
            r2 = results[key2][ii]
            if 'aggregated' in key1:
                r1 = pd.DataFrame({
                    'features': r1,
                    'abs_ranking': [-v for v in range(1, len(r1) + 1)],
                    'rank': np.arange(1, len(r1) + 1)
                })
            if 'aggregated' in key2:
                r2 = pd.DataFrame({
                    'features': r2,
                    'abs_ranking': [-v for v in range(1, len(r2) + 1)],
                    'rank': np.arange(1, len(r2) + 1)
                })
            rank1 = convert_term_ranking_to_feature_ranking(r1)
            rank2 = convert_term_ranking_to_feature_ranking(r2)
            c = rank_correlation(rank1, rank2).statistic
            corrs.append(c)
        correlation = np.mean(corrs)
        correla[(key1, key2)] = correlation
    print(dname, dataset)
    rf_correlations[dname] = correla

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

layout = [
        ["A", "A", "B", "B", "C", "C"],  # Row 1 (3 plots)
        ["D", "D", "E", "E", "F", "F"],  # Row 2 (3 plots)
        [".", "G", "G", "H", "H", "."]   # Row 3 (2 plots, centered)
    ]
axes_keys = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']
fig, ax = plt.subplot_mosaic(layout, figsize=(15, 15), constrained_layout=True, sharey=True)
name_data_mapping = {
    'Linear': 'Linear',
    'Piecewise': 'Piecewise',
    'Polynomial': 'Polynomial',
    'CNOHF': 'CNOHF',
    'COF': 'COF',
    'Photoswitch': 'Photoswitch',
    'Polymers': 'Polymers',
    'Redox': 'Redox',
}
order_mapping = {
    'Linear': 0,
    'Piecewise': 1,
    'Polynomial': 2,
    'CNOHF': 3,
    'Photoswitch': 4,
    'Polymers': 5,
    'Redox': 6,
    'COF': 7,
}
methods_mapping = {
    'lime': 'LIME',
    'shap': 'SHAP',
    'shapiq1': 'SHAP-IQ-1',
    'shapiq2': 'SHAP-IQ-2',
    'meg': 'MEG',
    'mmace': 'MMACE',
    'aggregated_mean': 'Mean',
    'aggregated_rra': 'RRA'
}
methods_order = ['LIME', 'SHAP', 'SHAP-IQ-1', 'SHAP-IQ-2', 'MMACE', 'MEG', 'Mean', 'RRA']
plt.style.use('default')
for i, (key, data) in enumerate(rf_correlations.items()):
    if key not in name_data_mapping:
        continue
    s = pd.Series(data)
    correlation_matrix = s.unstack()
    renamed_matrix = correlation_matrix.rename(index=methods_mapping, columns=methods_mapping)
    display(renamed_matrix)
    renamed_matrix = renamed_matrix[methods_order].loc[methods_order]
    axes_id = axes_keys[order_mapping[key]]
    mappable = sns.heatmap(renamed_matrix, annot=True, fmt=".1f", ax=ax[axes_id], cmap='coolwarm', vmin=-1, vmax=1, cbar=False, annot_kws={'size': 13})

    ax[axes_id].set_title(name_data_mapping[key], fontsize=16)
    ax[axes_id].tick_params(axis='both', which='major', labelsize=14)

ax['G'].tick_params(axis='y', labelleft=True, rotation=0)

#plt.tight_layout()
plt.savefig('../results/correlation_matrices_rf.pdf', dpi=300, bbox_inches='tight')
plt.show()